# Huấn luyện YOLOv8 cho nhận diện biển báo giao thông

Notebook này dùng dữ liệu đã chuẩn hóa theo định dạng YOLO, sau đó train model YOLOv8 với các kỹ thuật phù hợp cho bài toán phát hiện biển báo.

Mục tiêu:
- tải dữ liệu từ Hugging Face bằng `hf_transfer` với `force_download=True`;
- khởi tạo WandB theo dõi metrics qua từng epoch;
- lưu trọng số ở thư mục nội bộ `./runs/` hoặc `/kaggle/working/`;
- áp dụng cấu hình `P2` để cải thiện phát hiện vật thể nhỏ như biển báo.

In [ ]:
#%pip install ultralytics
#%pip install wandb

  Using cached ultralytics-8.4.137-py3-none-any.whl.metadata (45 kB)
  Using cached polars-1.44.1-py3-none-any.whl.metadata (11 kB)
  Using cached ultralytics_thop-2.1.6-py3-none-any.whl.metadata (13 kB)
  Using cached ultralytics_platform-0.1.20-py3-none-any.whl.metadata (8.4 kB)
  Using cached numpy-2.2.6-cp312-cp312-win_amd64.whl.metadata (60 kB)
Using cached ultralytics-8.4.137-py3-none-any.whl (1.4 MB)
Using cached numpy-2.2.6-cp312-cp312-win_amd64.whl (12.6 MB)
Using cached polars-1.44.1-py3-none-any.whl (865 kB)
Using cached ultralytics_platform-0.1.20-py3-none-any.whl (66 kB)
Using cached ultralytics_thop-2.1.6-py3-none-any.whl (30 kB)

   ---------------------------------------- 0/5 [polars]
   ---------------------------------------- 0/5 [polars]
   ---------------------------------------- 0/5 [polars]
   ---------------------------------------- 0/5 [polars]
   ---------------------------------------- 0/5 [polars]
   ---------------------------------------- 0/5 [polars]
   --

In [1]:
import os
import sys
from pathlib import Path

# Cài đặt gói hỗ trợ tải dataset từ Hugging Face.
# Dùng force_download=True để chắc chắn lấy bản mới nhất khi chạy lại notebook.
try:
    import hf_transfer  # noqa: F401
except ModuleNotFoundError:
    os.system('pip install hf_transfer')
    import hf_transfer  # noqa: F401

from huggingface_hub import hf_hub_download

REPO_ID = ""
FILE_NAME = "train_traffic_sign_dataset.json"
LOCAL_JSON_PATH = r"G:\HocKi9\Học Thống Kê\topic_final\final\archive\za_traffic_2020\traffic_train\train_traffic_sign_dataset.json"

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

# Notebook đang chạy trong thư mục notebooks, nên project_root sẽ là thư mục cha.
project_root = Path.cwd().resolve().parent
yolo_root = project_root / 'data' / 'yolo_format'
images_dir = yolo_root / 'images'
labels_dir = yolo_root / 'labels'

json_path = None

if REPO_ID:
    json_path = hf_hub_download(
        repo_id=REPO_ID,
        filename=FILE_NAME,
        repo_type='dataset',
        force_download=True,
    )

if json_path is None and os.path.exists(LOCAL_JSON_PATH):
    json_path = LOCAL_JSON_PATH

if json_path is None:
    raise FileNotFoundError('Không tìm thấy JSON gốc. Hãy điền REPO_ID hoặc cập nhật LOCAL_JSON_PATH.')

print(f'Bản JSON dataset: {json_path}')
print(f'Folder ảnh YOLO: {images_dir}')
print(f'Folder nhãn YOLO: {labels_dir}')
print(f'Thư mục data hiện có: {yolo_root.exists()}')

if not images_dir.exists() or not labels_dir.exists():
    raise FileNotFoundError('Thư mục data/yolo_format chưa được tạo. Hãy chạy notebook data_conversion.ipynb trước.')

Bản JSON dataset: G:\HocKi9\Học Thống Kê\topic_final\final\archive\za_traffic_2020\traffic_train\train_traffic_sign_dataset.json
Folder ảnh YOLO: G:\HocKi9\Học Thống Kê\topic_final\final\Object-Detection-Application\Traffic-Sign-Detection-ZaloAI\data\yolo_format\images
Folder nhãn YOLO: G:\HocKi9\Học Thống Kê\topic_final\final\Object-Detection-Application\Traffic-Sign-Detection-ZaloAI\data\yolo_format\labels
Thư mục data hiện có: True


In [2]:
from pathlib import Path
import textwrap

# 1. Tạo file data.yaml cho YOLO
# Xác định đường dẫn tuyệt đối (Absolute Path) chuẩn hóa
train_val_path = (project_root / 'data' / 'yolo_format' / 'images').as_posix()

# 1. Tạo file data.yaml cho YOLO (Đã sửa lỗi lặp thư mục data/data)
data_yaml = project_root / 'data' / 'traffic_sign_data.yaml'
data_yaml.write_text(textwrap.dedent(f"""\
    train: {train_val_path}
    val: {train_val_path}
    nc: 7
    names: ['no_entry', 'no_parkingwaiting', 'no_turning', 'max_speed', 'other_prohibition_signs', 'warning', 'mandatory']
"""), encoding='utf-8')

print(f'Đường dẫn data.yaml: {data_yaml}')
print(f'Nội dung YAML:\n{data_yaml.read_text(encoding="utf-8")}')

# 2. Tạo file cấu hình model YOLOv8-P2 CHUẨN KIẾN TRÚC (Đã sửa lỗi thiếu tham số)
configs_dir = project_root / 'configs'
configs_dir.mkdir(parents=True, exist_ok=True)
p2_cfg_path = configs_dir / 'yolov8_p2.yaml'

p2_cfg_path.write_text(textwrap.dedent("""\
    nc: 1
    scales:
      s: [0.33, 0.50, 1024]
    
    backbone:
      - [-1, 1, Conv, [64, 3, 2]]  # 0-P1/2
      - [-1, 1, Conv, [128, 3, 2]]  # 1-P2/4
      - [-1, 3, C2f, [128, True]]  # 2
      - [-1, 1, Conv, [256, 3, 2]]  # 3-P3/8
      - [-1, 6, C2f, [256, True]]  # 4
      - [-1, 1, Conv, [512, 3, 2]]  # 5-P4/16
      - [-1, 6, C2f, [512, True]]  # 6
      - [-1, 1, Conv, [1024, 3, 2]]  # 7-P5/32
      - [-1, 3, C2f, [1024, True]] # 8
      - [-1, 1, SPPF, [1024, 5]]  # 9
    
    head:
      - [-1, 1, Conv, [512, 1, 1]] # 10
      - [-1, 1, nn.Upsample, [None, 2, 'nearest']] # 11
      - [[-1, 6], 1, Concat, [1]]  # 12: Trộn với P4 (Đã thêm số 1)
      - [-1, 3, C2f, [512]]  # 13
    
      - [-1, 1, Conv, [256, 1, 1]] # 14
      - [-1, 1, nn.Upsample, [None, 2, 'nearest']] # 15
      - [[-1, 4], 1, Concat, [1]]  # 16: Trộn với P3 (Đã thêm số 1)
      - [-1, 3, C2f, [256]]  # 17 (P3/8-small)
    
      - [-1, 1, Conv, [128, 1, 1]] # 18
      - [-1, 1, nn.Upsample, [None, 2, 'nearest']] # 19
      - [[-1, 2], 1, Concat, [1]]  # 20: Trộn với P2 (Đã thêm số 1)
      - [-1, 3, C2f, [128]]  # 21 (P2/4-xsmall)
    
      - [[21, 17, 13, 9], 1, Detect, [nc]]  # 22: Đầu ra Detect (Đã thêm số 1)
"""), encoding='utf-8')

print(f'File cấu hình P2 đã sửa lỗi và tạo thành công tại: {p2_cfg_path}')

Đường dẫn data.yaml: G:\HocKi9\Học Thống Kê\topic_final\final\Object-Detection-Application\Traffic-Sign-Detection-ZaloAI\data\traffic_sign_data.yaml
Nội dung YAML:
train: G:/HocKi9/Học Thống Kê/topic_final/final/Object-Detection-Application/Traffic-Sign-Detection-ZaloAI/data/yolo_format/images
val: G:/HocKi9/Học Thống Kê/topic_final/final/Object-Detection-Application/Traffic-Sign-Detection-ZaloAI/data/yolo_format/images
nc: 7
names: ['no_entry', 'no_parkingwaiting', 'no_turning', 'max_speed', 'other_prohibition_signs', 'warning', 'mandatory']

File cấu hình P2 đã sửa lỗi và tạo thành công tại: G:\HocKi9\Học Thống Kê\topic_final\final\Object-Detection-Application\Traffic-Sign-Detection-ZaloAI\configs\yolov8_p2.yaml


In [3]:
#%pip install wandb
import os
import wandb

# Khởi tạo WandB ở chế độ offline nếu không có API key, để không crash khi môi trường không có token.
# Ở môi trường thật, ta có thể gán WANDB_API_KEY trong environment và chạy online.
wandb_key = os.getenv('WANDB_API_KEY')
wandb_mode = 'online' if wandb_key else 'offline'

wandb.init(
    project='traffic-sign-detection',
    name='yolov8-p2-train',
    group='yolov8',
    mode=wandb_mode,
    reinit=True,
    config={
        'dataset': 'traffic_sign_yolo',
        'model': 'yolov8-p2',
        'epochs': 80,
        'imgsz': 640,
        'batch_size': 16,
        'optimizer': 'auto',
        'notes': 'P2 layer + local weight save only',
    },
)

print(f'WandB mode: {wandb_mode}')
print('WandB đã được khởi tạo để theo dõi từng epoch.')

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


WandB mode: offline
WandB đã được khởi tạo để theo dõi từng epoch.


In [ ]:
import os
from pathlib import Path
import torch

# Không upload trọng số lên Hugging Face/Drive qua API.
# Chỉ lưu trực tiếp vào thư mục nội bộ để tránh crash phiên chạy.
save_root = Path('./runs').resolve()
save_root.mkdir(parents=True, exist_ok=True)

# Khởi tạo model YOLOv8 với cấu hình P2.
# phần này có thể thay bằng checkpoint pretrain phù hợp nếu đã có.
try:
    from ultralytics import YOLO
except ModuleNotFoundError:
    os.system('pip install ultralytics')
    from ultralytics import YOLO

model = YOLO(str(p2_cfg_path))

# Có thể dùng pretrained checkpoint nếu muốn warm start.
# model = YOLO('yolov8n.pt')

# Gọi hàm huấn luyện với ĐẦY ĐỦ các chiến thuật từ EDA
results = model.train(
    data=str(data_yaml),
    epochs=50, # origin:50
    imgsz=1280,  #origin:1280       # High-res để bảo toàn vật thể nhỏ
    batch=8,   #origin:8         # Hạ batch xuống 8 để tránh OOM với ảnh 1280
    #workers=0, #nhớ cmt dòng này đi, này để test trên local thôi
    project=str(save_root),
    name='yolov8s_p2_highres',
    device='0' if torch.cuda.is_available() else 'cpu',
    optimizer='AdamW',  # Sử dụng AdamW thay vì auto
    cos_lr=True,        # Cosine Annealing LR
    max_det=50,         # Tối ưu NMS theo mật độ vật thể
    iou=0.6,            # Giữ các biển báo đứng cạnh nhau
    cls=2.0,            # Trừng phạt nặng lỗi phân loại sai
    box=1.0,
    mosaic=1.0,         # Trộn ảnh cường độ cao
    translate=0.2,      # Dịch chuyển ảnh để trị Center Bias
    seed=42,
    verbose=True
)

# Lưu trọng số ở thư mục nội bộ, không upload qua API.
best_model_path = save_root / 'yolov8_p2_exp' / 'weights' / 'best.pt'
print(f'Đường dẫn mô hình tốt nhất: {best_model_path}')
print('File đã được lưu ở thư mục nội bộ, không upload API.')

wandb.finish()

WARNING no model scale passed. Assuming scale='s'.
Ultralytics 8.4.137  Python-3.12.7 torch-2.9.1+cpu CPU (11th Gen Intel Core i5-1135G7 @ 2.40GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=1.0, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=2.0, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=G:\HocKi9\Hc Thng K\topic_final\final\Object-Detection-Application\Traffic-Sign-Detection-ZaloAI\data\traffic_sign_data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.6, keras=False, kobj=1.0, line_width=None, lr0=0.01, lr